In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
!pip install -U transformers datasets peft accelerate bitsandbytes --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 142.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.6/503.6 kB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 17.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pylibcudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training
import torch

model_path = "google/gemma-3-1b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="eager",
    trust_remote_code=True
)

model = prepare_model_for_kbit_training(model)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

In [ ]:
import torch
from peft import get_peft_model, LoraConfig, TaskType

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 6,522,880 || all params: 1,006,408,832 || trainable%: 0.6481


In [ ]:
from datasets import Dataset
import json

with open("/content/train.jsonl", "r", encoding="utf-8") as f:
    data = [json.loads(line) for line in f if line.strip()]

dataset = Dataset.from_list(data)

for i in range(min(3, len(dataset))):
    print(json.dumps(dataset[i], indent=2, ensure_ascii=False))

{
  "messages": [
    {
      "content": "Kenapa bahasa PHP sering dihujat dan dijadikan meme di kalangan programmer?",
      "role": "user"
    },
    {
      "content": "PHP sering dihujat dan dijadikan bahan meme di kalangan programmer bukan karena PHP jelek secara objektif, tapi karena kombinasi dari **warisan sejarahnya, desain awalnya, dan ekspektasi modern yang berubah**. Berikut ini adalah penjelasan detailnya:\n\n\n## 🧠 1. Desain Awal PHP Kurang Konsisten\n\nPHP awalnya dibuat sebagai *Personal Home Page Tools* oleh Rasmus Lerdorf di tahun 1994, bukan sebagai bahasa pemrograman besar seperti sekarang. Karena itu:\n\n* Nama fungsi sering tidak konsisten: contoh `strtoupper()` vs `strlen()` (kadang pakai prefix `str`, kadang tidak).\n* Urutan argumen fungsi bisa membingungkan: misalnya `array_map(function, array)` vs `strpos(haystack, needle)`.\n\n> 👉 Banyak fungsi terlihat “asal jadi”, karena dulunya memang bukan untuk skala besar.\n\n\n## 🧨 2. Kode Legacy PHP Banyak yang Buruk

In [ ]:
from transformers import DataCollatorForLanguageModeling

def tokenize(example):
    return tokenizer(
        tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False
        ),
        truncation=True,
        max_length=4096,
    )

tokenized_dataset = dataset.map(tokenize, remove_columns=dataset.column_names)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

lengths = [
    len(tokenizer.apply_chat_template(x["messages"], tokenize=True))
    for x in dataset
]
print("Max length:", max(lengths))
print("Avg length:", sum(lengths) // len(lengths))
print(tokenizer.apply_chat_template(dataset[0]["messages"], tokenize=False))

Map:   0%|          | 0/1400 [00:00<?, ? examples/s]

Max length: 3154
Avg length: 424
<bos><start_of_turn>user
Kenapa bahasa PHP sering dihujat dan dijadikan meme di kalangan programmer?<end_of_turn>
<start_of_turn>model
PHP sering dihujat dan dijadikan bahan meme di kalangan programmer bukan karena PHP jelek secara objektif, tapi karena kombinasi dari **warisan sejarahnya, desain awalnya, dan ekspektasi modern yang berubah**. Berikut ini adalah penjelasan detailnya:


## 🧠 1. Desain Awal PHP Kurang Konsisten

PHP awalnya dibuat sebagai *Personal Home Page Tools* oleh Rasmus Lerdorf di tahun 1994, bukan sebagai bahasa pemrograman besar seperti sekarang. Karena itu:

* Nama fungsi sering tidak konsisten: contoh `strtoupper()` vs `strlen()` (kadang pakai prefix `str`, kadang tidak).
* Urutan argumen fungsi bisa membingungkan: misalnya `array_map(function, array)` vs `strpos(haystack, needle)`.

> 👉 Banyak fungsi terlihat “asal jadi”, karena dulunya memang bukan untuk skala besar.


## 🧨 2. Kode Legacy PHP Banyak yang Buruk

Karena PHP sang

In [ ]:
torch.cuda.empty_cache()

In [ ]:
from transformers import TrainingArguments, Trainer

model.gradient_checkpointing_enable()

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/output",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    warmup_ratio=0.1,
    learning_rate=2e-4,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    logging_dir="logs",
    logging_steps=50,
    save_steps=250,
    save_total_limit=1,
    report_to="none",
    fp16=True,
    group_by_length=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

trainer.train()


/tmp/ipython-input-1341178210.py:24: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'pad_token_id': 1}.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
50,3.450800
100,2.388000
150,2.234900
200,2.156700
250,1.987600
300,2.046200
350,2.052500
400,1.889500
450,1.881300
500,1.832800


TrainOutput(global_step=525, training_loss=2.17928963070824, metrics={'train_runtime': 2863.7175, 'train_samples_per_second': 1.467, 'train_steps_per_second': 0.183, 'total_flos': 7558134910380288.0, 'train_loss': 2.17928963070824, 'epoch': 3.0})

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-1b-it", trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-3-1b-it",
    device_map="auto",
    torch_dtype=torch.float16,
    attn_implementation="eager",
    trust_remote_code=True
)

lora_model = PeftModel.from_pretrained(
    base_model,
    "/content/drive/MyDrive/output/checkpoint-525",
    is_trainable=False
)

merged_model = lora_model.merge_and_unload()

save_path = "/content/drive/MyDrive/Sorachio-1B-4096-2e-4"
merged_model.save_pretrained(save_path, safe_serialization=True)
tokenizer.save_pretrained(save_path)

`torch_dtype` is deprecated! Use `dtype` instead!


('/content/drive/MyDrive/Sorachio-1B-4096-2e-4/tokenizer_config.json',
 '/content/drive/MyDrive/Sorachio-1B-4096-2e-4/special_tokens_map.json',
 '/content/drive/MyDrive/Sorachio-1B-4096-2e-4/chat_template.jinja',
 '/content/drive/MyDrive/Sorachio-1B-4096-2e-4/tokenizer.model',
 '/content/drive/MyDrive/Sorachio-1B-4096-2e-4/added_tokens.json',
 '/content/drive/MyDrive/Sorachio-1B-4096-2e-4/tokenizer.json')

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_path = "/content/drive/MyDrive/Sorachio-1B-4096-2e-4"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto",
    torch_dtype=torch.float16,
    attn_implementation="eager"
).eval()

messages = [
    {"role": "user", "content": "Perkenalkan dirimu"}
]

input_ids = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=(input_ids != tokenizer.pad_token_id).long(),
        max_new_tokens=256,
        do_sample=True,
        top_p=0.9,
        temperature=0.6,
        pad_token_id=tokenizer.eos_token_id
    )

output_text = tokenizer.decode(outputs[0][input_ids.shape[-1]:], skip_special_tokens=True)
print(output_text)

Halo! Senang sekali bertemu dengan kamu.
Aku Sorachio, asisten AI yang dirancang untuk memahami dan merespons pertanyaan kamu dengan cara yang ramah, informatif, dan mudah dimengerti.
Aku bisa membantu kamu dalam berbagai hal, mulai dari menjawab pertanyaan umum, memberikan informasi, hingga membuat percakapan yang menyenangkan.
Kalau kamu butuh bantuan, jangan ragu untuk bertanya ya! 😄
